In [0]:
%sql
-- Set up widget for mode parameter
CREATE WIDGET TEXT mode DEFAULT 'full_refresh';

-- Set up schema
CREATE SCHEMA IF NOT EXISTS workspace.medallion_sql_pipeline;
USE workspace.medallion_sql_pipeline;

SELECT 'Pipeline initialized' AS status, '${mode}' AS mode;

In [0]:
%sql
-- Create watermark and log tables
CREATE TABLE IF NOT EXISTS workspace.medallion_sql_pipeline.pipeline_watermarks (
    table_name      STRING,
    last_order_date DATE,
    rows_processed  BIGINT,
    run_status      STRING,
    run_mode        STRING,
    updated_at      TIMESTAMP
)
USING DELTA;

CREATE TABLE IF NOT EXISTS workspace.medallion_sql_pipeline.pipeline_run_log (
    run_id       STRING,
    run_mode     STRING,
    stage        STRING,
    rows_in      BIGINT,
    rows_out     BIGINT,
    status       STRING,
    error_msg    STRING,
    started_at   TIMESTAMP,
    finished_at  TIMESTAMP
)
USING DELTA;

SELECT 'Watermark tables ready' AS status;

In [0]:
%sql
-- Load Customer dimension
CREATE OR REPLACE TABLE workspace.medallion_sql_pipeline.bronze_customer
USING DELTA AS
SELECT 
  `Customer ID` AS Customer_ID,
  Customer,
  City,
  `State-Province` AS State_Province,
  `Country-Region` AS Country_Region,
  `Postal Code` AS Postal_Code,
  _source,
  CURRENT_TIMESTAMP() AS _ingest_time,
  'RAW_Customer.csv' AS _source_file
FROM read_files(
  '/Volumes/workspace/default/course_data/raw_data/RAW_Customer.csv',
  format => 'csv', header => true, multiLine => true, escape => '"'
);

-- Load Product dimension
CREATE OR REPLACE TABLE workspace.medallion_sql_pipeline.bronze_product
USING DELTA AS
SELECT 
  SKU,
  Product,
  `Standard Cost` AS Standard_Cost,
  Color,
  `List Price` AS List_Price,
  Model,
  Subcategory,
  Category,
  CURRENT_TIMESTAMP() AS _ingest_time,
  'RAW_Product.csv' AS _source_file
FROM read_files(
  '/Volumes/workspace/default/course_data/raw_data/RAW_Product.csv',
  format => 'csv', header => true, multiLine => true, escape => '"'
);

-- Load Reseller dimension
CREATE OR REPLACE TABLE workspace.medallion_sql_pipeline.bronze_reseller
USING DELTA AS
SELECT 
  `Reseller ID` AS Reseller_ID,
  `Business Type` AS Business_Type,
  Reseller,
  City,
  `State-Province` AS State_Province,
  `Country-Region` AS Country_Region,
  `Postal Code` AS Postal_Code,
  CURRENT_TIMESTAMP() AS _ingest_time,
  'RAW_Reseller.csv' AS _source_file
FROM read_files(
  '/Volumes/workspace/default/course_data/raw_data/RAW_Reseller.csv',
  format => 'csv', header => true, multiLine => true, escape => '"'
);

-- Load Sales Territory dimension
CREATE OR REPLACE TABLE workspace.medallion_sql_pipeline.bronze_sales_territory
USING DELTA AS
SELECT 
  REGION AS Region,
  Country,
  Sales_Group,
  notes,
  CURRENT_TIMESTAMP() AS _ingest_time,
  'RAW_SalesTerritory.csv' AS _source_file
FROM read_files(
  '/Volumes/workspace/default/course_data/raw_data/RAW_SalesTerritory.csv',
  format => 'csv', header => true, multiLine => true, escape => '"'
);

-- Load Date dimension
CREATE OR REPLACE TABLE workspace.medallion_sql_pipeline.bronze_date
USING DELTA AS
SELECT 
  Date,
  `Fiscal Year` AS Fiscal_Year,
  `Fiscal Quarter` AS Fiscal_Quarter,
  Month,
  `Full Date` AS Full_Date,
  CURRENT_TIMESTAMP() AS _ingest_time,
  'RAW_Date.csv' AS _source_file
FROM read_files(
  '/Volumes/workspace/default/course_data/raw_data/RAW_Date.csv',
  format => 'csv', header => true, multiLine => true, escape => '"'
);

-- Load Sales Order dimension
CREATE OR REPLACE TABLE workspace.medallion_sql_pipeline.bronze_sales_order
USING DELTA AS
SELECT 
  Channel,
  `Sales Order` AS Sales_Order,
  `Sales Order Line` AS Sales_Order_Line,
  created_date,
  CURRENT_TIMESTAMP() AS _ingest_time,
  'RAW_SalesOrder.csv' AS _source_file
FROM read_files(
  '/Volumes/workspace/default/course_data/raw_data/RAW_SalesOrder.csv',
  format => 'csv', header => true, multiLine => true, escape => '"'
);

SELECT 'Dimensions loaded' AS status,
  (SELECT COUNT(*) FROM bronze_customer) AS customers,
  (SELECT COUNT(*) FROM bronze_product) AS products,
  (SELECT COUNT(*) FROM bronze_reseller) AS resellers,
  (SELECT COUNT(*) FROM bronze_sales_territory) AS territories,
  (SELECT COUNT(*) FROM bronze_date) AS dates,
  (SELECT COUNT(*) FROM bronze_sales_order) AS orders;

In [0]:
%sql
-- Get watermark for incremental mode
CREATE OR REPLACE TEMPORARY VIEW watermark AS
SELECT 
  COALESCE(MAX(last_order_date), DATE('1900-01-01')) AS wm_date
FROM workspace.medallion_sql_pipeline.pipeline_watermarks
WHERE table_name = 'bronze_sales' AND run_status = 'SUCCESS';

-- Create table if not exists
CREATE TABLE IF NOT EXISTS workspace.medallion_sql_pipeline.bronze_sales (
  Channel STRING,
  Sales_Order STRING,
  Sales_Order_Line STRING,
  Customer_ID STRING,
  Reseller_ID STRING,
  SKU STRING,
  Region STRING,
  Country STRING,
  Order_Date STRING,
  Due_Date STRING,
  Ship_Date STRING,
  Order_Quantity STRING,
  Unit_Price STRING,
  Unit_Price_Discount_Pct STRING,
  Product_Standard_Cost STRING,
  Total_Product_Cost STRING,
  Extended_Amount STRING,
  Sales_Amount STRING,
  _batch_id STRING,
  _ingest_time TIMESTAMP,
  _source_file STRING
)
USING DELTA;

-- For full refresh: delete all existing data
DELETE FROM workspace.medallion_sql_pipeline.bronze_sales 
WHERE LOWER(TRIM('${mode}')) = 'full_refresh';

-- Prepare new data based on mode
CREATE OR REPLACE TEMPORARY VIEW new_sales_data AS
SELECT 
  raw.Channel,
  raw.`Sales Order` AS Sales_Order,
  raw.`Sales Order Line` AS Sales_Order_Line,
  raw.`Customer ID` AS Customer_ID,
  raw.`Reseller ID` AS Reseller_ID,
  raw.SKU,
  raw.Region,
  raw.Country,
  raw.`Order Date` AS Order_Date,
  raw.`Due Date` AS Due_Date,
  raw.`Ship Date` AS Ship_Date,
  raw.`Order Quantity` AS Order_Quantity,
  raw.`Unit Price` AS Unit_Price,
  raw.`Unit Price Discount Pct` AS Unit_Price_Discount_Pct,
  raw.`Product Standard Cost` AS Product_Standard_Cost,
  raw.`Total Product Cost` AS Total_Product_Cost,
  raw.`Extended Amount` AS Extended_Amount,
  raw.`Sales Amount` AS Sales_Amount,
  raw._batch_id,
  CURRENT_TIMESTAMP() AS _ingest_time,
  'RAW_Sales.csv' AS _source_file
FROM read_files(
  '/Volumes/workspace/default/course_data/raw_data/RAW_Sales.csv',
  format => 'csv', header => true, multiLine => true, escape => '"'
) raw
CROSS JOIN watermark wm
WHERE 
  -- Full refresh: load all data
  (LOWER(TRIM('${mode}')) = 'full_refresh')
  OR
  -- Incremental: load only new data after watermark
  (LOWER(TRIM('${mode}')) = 'incremental'
   AND COALESCE(TRY_TO_DATE(raw.`Order Date`, 'yyyy-MM-dd'), TRY_TO_DATE(raw.`Order Date`, 'MM/dd/yyyy')) > wm.wm_date);

-- Insert new data (works for both modes)
INSERT INTO workspace.medallion_sql_pipeline.bronze_sales
SELECT * FROM new_sales_data;

-- Update watermark
INSERT INTO workspace.medallion_sql_pipeline.pipeline_watermarks
SELECT 
  'bronze_sales' AS table_name,
  MAX(COALESCE(TRY_TO_DATE(Order_Date, 'yyyy-MM-dd'), TRY_TO_DATE(Order_Date, 'MM/dd/yyyy'))) AS last_order_date,
  COUNT(*) AS rows_processed,
  'SUCCESS' AS run_status,
  LOWER(TRIM('${mode}')) AS run_mode,
  CURRENT_TIMESTAMP() AS updated_at
FROM workspace.medallion_sql_pipeline.bronze_sales;

-- Log the run
INSERT INTO workspace.medallion_sql_pipeline.pipeline_run_log
SELECT
  DATE_FORMAT(CURRENT_TIMESTAMP(), 'yyyyMMddHHmmss') AS run_id,
  LOWER(TRIM('${mode}')) AS run_mode,
  'bronze_sales' AS stage,
  0 AS rows_in,
  COUNT(*) AS rows_out,
  'SUCCESS' AS status,
  '' AS error_msg,
  CURRENT_TIMESTAMP() AS started_at,
  CURRENT_TIMESTAMP() AS finished_at
FROM workspace.medallion_sql_pipeline.bronze_sales;

SELECT 
  'Sales fact loaded' AS status, 
  LOWER(TRIM('${mode}')) AS mode,
  COUNT(*) AS total_rows,
  (SELECT wm_date FROM watermark) AS watermark_date
FROM bronze_sales;

In [0]:
%sql
-- Create Silver Customer Dimension
CREATE OR REPLACE TABLE workspace.medallion_sql_pipeline.silver_dim_customer
USING DELTA AS
SELECT 
    ROW_NUMBER() OVER(ORDER BY Customer_ID) AS CustomerKey,
    Customer_ID AS CustomerNaturalKey,
    COALESCE(NULLIF(TRIM(Customer), ''), 'Unknown') AS CustomerName,
    COALESCE(NULLIF(TRIM(City), ''), 'Unknown') AS City,
    INITCAP(TRIM(State_Province)) AS StateProvince,
    INITCAP(TRIM(Country_Region)) AS CountryRegion,
    COALESCE(NULLIF(TRIM(Postal_Code), ''), '00000') AS PostalCode
FROM (
    SELECT DISTINCT 
        Customer_ID, Customer, City, State_Province, Country_Region, Postal_Code 
    FROM bronze_customer 
    WHERE Customer_ID IS NOT NULL
);

SELECT 'Customer dimension created' AS status, COUNT(*) AS rows FROM silver_dim_customer;

In [0]:
%sql
-- Create Silver Product Dimension
CREATE OR REPLACE TABLE workspace.medallion_sql_pipeline.silver_dim_product
USING DELTA AS
SELECT 
    ROW_NUMBER() OVER(ORDER BY SKU) AS ProductKey, 
    SKU,
    TRIM(FIRST(Product)) AS Product,
    TRIM(FIRST(Model)) AS Model,
    COALESCE(INITCAP(NULLIF(TRIM(FIRST(Category)), '')), 'Unknown') AS Category,
    INITCAP(TRIM(FIRST(Subcategory))) AS Subcategory,
    COALESCE(INITCAP(NULLIF(TRIM(FIRST(Color)), '')), 'No Color') AS Color,
    TRY_CAST(REGEXP_REPLACE(FIRST(List_Price), '[^\\d\\.\\-]', '') AS DOUBLE) AS ListPrice,
    COALESCE(TRY_CAST(REGEXP_REPLACE(FIRST(Standard_Cost), '[^\\d\\.\\-]', '') AS DOUBLE), 0.0) AS StandardCost
FROM bronze_product 
WHERE SKU IS NOT NULL
GROUP BY SKU
HAVING TRY_CAST(REGEXP_REPLACE(FIRST(List_Price), '[^\\d\\.\\-]', '') AS DOUBLE) > 0;

SELECT 'Product dimension created' AS status, COUNT(*) AS rows FROM silver_dim_product;

In [0]:
%sql
-- Create Silver Reseller Dimension
CREATE OR REPLACE TABLE workspace.medallion_sql_pipeline.silver_dim_reseller
USING DELTA AS
SELECT 
    ROW_NUMBER() OVER(ORDER BY Reseller_ID) AS ResellerKey,
    Reseller_ID AS ResellerNaturalKey, 
    TRIM(Reseller) AS Reseller,
    COALESCE(INITCAP(NULLIF(TRIM(Business_Type), '')), 'Unknown') AS BusinessType,
    COALESCE(NULLIF(TRIM(City), ''), 'Unknown') AS City,
    INITCAP(TRIM(State_Province)) AS StateProvince,
    INITCAP(TRIM(Country_Region)) AS CountryRegion,
    COALESCE(NULLIF(TRIM(Postal_Code), ''), '00000') AS PostalCode
FROM (
    SELECT DISTINCT 
        Reseller_ID, Reseller, Business_Type, City, State_Province, Country_Region, Postal_Code 
    FROM bronze_reseller 
    WHERE Reseller_ID IS NOT NULL
);

SELECT 'Reseller dimension created' AS status, COUNT(*) AS rows FROM silver_dim_reseller;

In [0]:
%sql
-- Create Silver Territory Dimension
CREATE OR REPLACE TABLE workspace.medallion_sql_pipeline.silver_dim_territory
USING DELTA AS
SELECT 
    ROW_NUMBER() OVER(ORDER BY Region) AS SalesTerritoryKey,
    INITCAP(TRIM(Region)) AS Region, 
    INITCAP(TRIM(Country)) AS Country, 
    INITCAP(TRIM(Sales_Group)) AS SalesGroup
FROM (
    SELECT DISTINCT Region, Country, Sales_Group 
    FROM bronze_sales_territory 
    WHERE Region IS NOT NULL
);

-- Create Silver Date Dimension
CREATE OR REPLACE TABLE workspace.medallion_sql_pipeline.silver_dim_date
USING DELTA AS
WITH dates AS (
    SELECT DISTINCT 
        COALESCE(
            TRY_TO_DATE(Date, 'yyyy-MM-dd'), 
            TRY_TO_DATE(Date, 'MM/dd/yyyy'), 
            TRY_TO_DATE(Date, 'dd-MM-yyyy'), 
            TRY_TO_DATE(Date, 'yyyyMMdd')
        ) AS Date,
        Fiscal_Year, 
        Fiscal_Quarter, 
        Month
    FROM bronze_date 
    WHERE Date IS NOT NULL
)
SELECT 
    TRY_CAST(DATE_FORMAT(Date, 'yyyyMMdd') AS INT) AS DateKey, 
    Date,
    CONCAT('FY', REGEXP_EXTRACT(Fiscal_Year, '(\\d{4})', 1)) AS FiscalYear,
    COALESCE(
        Fiscal_Quarter, 
        CONCAT('FY', REGEXP_EXTRACT(Fiscal_Year, '(\\d{4})', 1), ' Q', 
               CAST(CEIL((MONTH(Date) + CASE WHEN MONTH(Date) >= 7 THEN -6 ELSE 6 END) / 3.0) AS INT))
    ) AS FiscalQuarter,
    Month, 
    TRY_CAST(DATE_FORMAT(Date, 'yyyyMM') AS INT) AS MonthKey
FROM dates 
WHERE Date IS NOT NULL;

SELECT 'Territory & Date dimensions created' AS status,
    (SELECT COUNT(*) FROM silver_dim_territory) AS territories,
    (SELECT COUNT(*) FROM silver_dim_date) AS dates;

In [0]:
%sql
-- Create Silver Sales Order Dimension
CREATE OR REPLACE TABLE workspace.medallion_sql_pipeline.silver_dim_salesorder
USING DELTA AS
SELECT 
    TRY_CAST(REGEXP_EXTRACT(SalesOrder, 'SO(\\d+)', 1) AS BIGINT) * 100 + 
    TRY_CAST(REGEXP_EXTRACT(SalesOrderLine, '-\\s*(\\d+)$', 1) AS BIGINT) AS SalesOrderLineKey,
    SalesOrder, 
    SalesOrderLine, 
    Channel
FROM (
    SELECT DISTINCT 
        TRIM(Sales_Order) AS SalesOrder, 
        TRIM(Sales_Order_Line) AS SalesOrderLine, 
        COALESCE(INITCAP(TRIM(Channel)), 'Unknown') AS Channel 
    FROM bronze_sales_order 
    WHERE Sales_Order_Line IS NOT NULL
);

SELECT 'Sales Order dimension created' AS status, COUNT(*) AS rows FROM silver_dim_salesorder;

In [0]:
%sql
-- Create Silver Sales Fact
CREATE OR REPLACE TABLE workspace.medallion_sql_pipeline.silver_fact_sales
USING DELTA AS
WITH dedup_sales AS (
    SELECT 
        *, 
        ROW_NUMBER() OVER(
            PARTITION BY Sales_Order_Line 
            ORDER BY Order_Date DESC
        ) AS rn 
    FROM bronze_sales
)
SELECT 
    so.SalesOrderLineKey, 
    c.CustomerKey, 
    p.ProductKey, 
    r.ResellerKey, 
    t.SalesTerritoryKey,
    TRY_CAST(
        DATE_FORMAT(
            COALESCE(
                TRY_TO_DATE(s.Order_Date, 'yyyy-MM-dd'), 
                TRY_TO_DATE(s.Order_Date, 'MM/dd/yyyy')
            ), 
            'yyyyMMdd'
        ) AS INT
    ) AS OrderDateKey,
    INITCAP(TRIM(s.Channel)) AS Channel,
    TRY_CAST(ROUND(TRY_CAST(s.Order_Quantity AS DOUBLE), 0) AS INT) AS OrderQuantity,
    TRY_CAST(s.Unit_Price AS DOUBLE) AS UnitPrice,
    CASE 
        WHEN s.Unit_Price_Discount_Pct LIKE '%\\%%' THEN 
            TRY_CAST(REGEXP_EXTRACT(s.Unit_Price_Discount_Pct, '([\\d\\.]+)%', 1) AS DOUBLE) / 100.0 
        ELSE 
            TRY_CAST(s.Unit_Price_Discount_Pct AS DOUBLE) 
    END AS UnitPriceDiscountPct,
    TRY_CAST(s.Product_Standard_Cost AS DOUBLE) AS ProductStandardCost,
    TRY_CAST(s.Total_Product_Cost AS DOUBLE) AS TotalProductCost,
    TRY_CAST(s.Extended_Amount AS DOUBLE) AS ExtendedAmount,
    TRY_CAST(s.Sales_Amount AS DOUBLE) AS SalesAmount,
    COALESCE(
        TRY_TO_DATE(s.Order_Date, 'yyyy-MM-dd'), 
        TRY_TO_DATE(s.Order_Date, 'MM/dd/yyyy')
    ) AS OrderDate,
    COALESCE(
        TRY_TO_DATE(s.Due_Date, 'yyyy-MM-dd'), 
        TRY_TO_DATE(s.Due_Date, 'MM/dd/yyyy')
    ) AS DueDate,
    COALESCE(
        TRY_TO_DATE(s.Ship_Date, 'yyyy-MM-dd'), 
        TRY_TO_DATE(s.Ship_Date, 'MM/dd/yyyy')
    ) AS ShipDate,
    CASE 
        WHEN TRY_CAST(s.Sales_Amount AS DOUBLE) < 0 THEN 1 
        ELSE 0 
    END AS IsReturn
FROM dedup_sales s
LEFT JOIN silver_dim_customer c ON TRIM(s.Customer_ID) = c.CustomerNaturalKey
LEFT JOIN silver_dim_product p ON TRIM(s.SKU) = p.SKU
LEFT JOIN silver_dim_reseller r ON TRIM(s.Reseller_ID) = r.ResellerNaturalKey
LEFT JOIN silver_dim_territory t ON INITCAP(TRIM(s.Region)) = t.Region AND INITCAP(TRIM(s.Country)) = t.Country
LEFT JOIN silver_dim_salesorder so ON TRIM(s.Sales_Order_Line) = so.SalesOrderLine
WHERE s.rn = 1;

-- Update watermark
INSERT INTO workspace.medallion_sql_pipeline.pipeline_watermarks
SELECT 
  'silver_fact_sales' AS table_name,
  MAX(OrderDate) AS last_order_date,
  COUNT(*) AS rows_processed,
  'SUCCESS' AS run_status,
  LOWER(TRIM('${mode}')) AS run_mode,
  CURRENT_TIMESTAMP() AS updated_at
FROM silver_fact_sales;

-- Log the run
INSERT INTO workspace.medallion_sql_pipeline.pipeline_run_log
SELECT
  DATE_FORMAT(CURRENT_TIMESTAMP(), 'yyyyMMddHHmmss') AS run_id,
  LOWER(TRIM('${mode}')) AS run_mode,
  'silver_sales' AS stage,
  0 AS rows_in,
  COUNT(*) AS rows_out,
  'SUCCESS' AS status,
  '' AS error_msg,
  CURRENT_TIMESTAMP() AS started_at,
  CURRENT_TIMESTAMP() AS finished_at
FROM silver_fact_sales;

SELECT 
  'Sales fact created' AS status, 
  LOWER(TRIM('${mode}')) AS mode,
  COUNT(*) AS rows 
FROM silver_fact_sales;

In [0]:
%sql
-- Create Gold Sales by Month table
CREATE OR REPLACE TABLE workspace.medallion_sql_pipeline.gold_sales_by_month
USING DELTA AS
WITH base AS (
    SELECT 
        d.FiscalYear, 
        d.FiscalQuarter, 
        d.MonthKey, 
        t.Region, 
        t.Country, 
        t.SalesGroup, 
        p.Category, 
        f.Channel,
        COUNT(f.SalesOrderLineKey) AS OrderLines, 
        SUM(f.OrderQuantity) AS TotalUnits,
        ROUND(SUM(f.SalesAmount), 2) AS TotalRevenue, 
        ROUND(AVG(f.SalesAmount), 2) AS AvgOrderValue,
        ROUND(SUM(f.TotalProductCost), 2) AS TotalCost, 
        COUNT(DISTINCT f.CustomerKey) AS UniqueCustomers
    FROM silver_fact_sales f
    JOIN silver_dim_date d ON f.OrderDateKey = d.DateKey
    LEFT JOIN silver_dim_territory t ON f.SalesTerritoryKey = t.SalesTerritoryKey
    LEFT JOIN silver_dim_product p ON f.ProductKey = p.ProductKey
    WHERE f.IsReturn = 0
    GROUP BY d.FiscalYear, d.FiscalQuarter, d.MonthKey, t.Region, t.Country, t.SalesGroup, p.Category, f.Channel
)
SELECT 
    *, 
    ROUND(TotalRevenue - TotalCost, 2) AS GrossProfit,
    CASE 
        WHEN TotalRevenue != 0 THEN 
            ROUND(((TotalRevenue - TotalCost) / TotalRevenue) * 100, 1) 
    END AS GrossMarginPct,
    CURRENT_TIMESTAMP() AS _gold_timestamp
FROM base;

SELECT 'Sales by month created' AS status, COUNT(*) AS rows FROM gold_sales_by_month;

In [0]:
%sql
-- Create Gold Product Ranking table
CREATE OR REPLACE TABLE workspace.medallion_sql_pipeline.gold_product_ranking
USING DELTA AS
WITH tr AS (
    SELECT SUM(SalesAmount) AS GrandTotal 
    FROM silver_fact_sales 
    WHERE IsReturn = 0
),
agg AS (
    SELECT 
        p.ProductKey, 
        p.SKU, 
        p.Product, 
        p.Model, 
        p.Category, 
        p.Subcategory, 
        p.Color,
        SUM(f.OrderQuantity) AS UnitsSold, 
        ROUND(SUM(f.SalesAmount), 2) AS TotalRevenue, 
        ROUND(SUM(f.TotalProductCost), 2) AS TotalCost,
        ROUND(AVG(f.UnitPrice), 2) AS AvgSellingPrice, 
        COUNT(f.SalesOrderLineKey) AS OrderLines
    FROM silver_fact_sales f 
    LEFT JOIN silver_dim_product p ON f.ProductKey = p.ProductKey
    WHERE f.IsReturn = 0 
    GROUP BY p.ProductKey, p.SKU, p.Product, p.Model, p.Category, p.Subcategory, p.Color
)
SELECT 
    a.*, 
    ROUND(a.TotalRevenue - a.TotalCost, 2) AS GrossProfit,
    CASE 
        WHEN a.TotalRevenue != 0 THEN 
            ROUND(((a.TotalRevenue - a.TotalCost) / a.TotalRevenue) * 100, 1) 
    END AS GrossMarginPct,
    CASE 
        WHEN tr.GrandTotal != 0 THEN 
            ROUND((a.TotalRevenue / tr.GrandTotal) * 100, 2) 
    END AS RevSharePct,
    RANK() OVER(ORDER BY a.TotalRevenue DESC) AS RevenueRank,
    CURRENT_TIMESTAMP() AS _gold_timestamp
FROM agg a 
CROSS JOIN tr;

SELECT 'Product ranking created' AS status, COUNT(*) AS rows FROM gold_product_ranking;

In [0]:
%sql
-- Create Gold Customer Summary table
CREATE OR REPLACE TABLE workspace.medallion_sql_pipeline.gold_customer_summary
USING DELTA AS
WITH rfm AS (
    SELECT 
        CustomerKey, 
        DATEDIFF(CURRENT_DATE(), MAX(OrderDate)) AS RecencyDays, 
        COUNT(SalesOrderLineKey) AS Frequency,
        ROUND(SUM(SalesAmount), 2) AS LifetimeValue, 
        ROUND(AVG(SalesAmount), 2) AS AvgOrderValue, 
        MAX(OrderDate) AS LastOrderDate
    FROM silver_fact_sales 
    WHERE IsReturn = 0 AND CustomerKey IS NOT NULL 
    GROUP BY CustomerKey
),
scores AS (
    SELECT 
        *,
        CASE 
            WHEN RecencyDays <= 30 THEN 5 
            WHEN RecencyDays <= 90 THEN 4 
            WHEN RecencyDays <= 180 THEN 3 
            WHEN RecencyDays <= 365 THEN 2 
            ELSE 1 
        END AS R,
        CASE 
            WHEN Frequency >= 20 THEN 5 
            WHEN Frequency >= 10 THEN 4 
            WHEN Frequency >= 5 THEN 3 
            WHEN Frequency >= 2 THEN 2 
            ELSE 1 
        END AS F,
        CASE 
            WHEN LifetimeValue >= 20000 THEN 5 
            WHEN LifetimeValue >= 10000 THEN 4 
            WHEN LifetimeValue >= 3000 THEN 3 
            WHEN LifetimeValue >= 500 THEN 2 
            ELSE 1 
        END AS M
    FROM rfm
)
SELECT 
    c.CustomerKey, 
    c.CustomerNaturalKey, 
    c.CustomerName, 
    c.City, 
    c.StateProvince, 
    c.CountryRegion,
    COALESCE(s.RecencyDays, 9999) AS RecencyDays, 
    COALESCE(s.Frequency, 0) AS Frequency, 
    COALESCE(s.LifetimeValue, 0.0) AS LifetimeValue, 
    s.AvgOrderValue, 
    s.LastOrderDate,
    CASE 
        WHEN (R+F+M) >= 13 THEN 'Champions' 
        WHEN (R+F+M) >= 10 THEN 'Loyal' 
        WHEN (R+F+M) >= 7 THEN 'Potential' 
        WHEN (R+F+M) >= 5 THEN 'At Risk' 
        ELSE 'Lost' 
    END AS CustomerSegment,
    CURRENT_TIMESTAMP() AS _gold_timestamp
FROM silver_dim_customer c 
LEFT JOIN scores s ON c.CustomerKey = s.CustomerKey;

SELECT 'Customer summary created' AS status, COUNT(*) AS rows FROM gold_customer_summary;

In [0]:
%sql
-- Create Gold Channel Compare table
CREATE OR REPLACE TABLE workspace.medallion_sql_pipeline.gold_channel_compare
USING DELTA AS
SELECT 
    d.FiscalYear, 
    f.Channel, 
    p.Category, 
    COUNT(f.SalesOrderLineKey) AS OrderLines, 
    SUM(f.OrderQuantity) AS UnitsSold,
    ROUND(SUM(f.SalesAmount), 2) AS TotalRevenue, 
    ROUND(AVG(f.SalesAmount), 2) AS AvgOrderValue, 
    ROUND(SUM(f.TotalProductCost), 2) AS TotalCost, 
    COUNT(DISTINCT f.CustomerKey) AS UniqueCustomers,
    ROUND(SUM(f.SalesAmount) - SUM(f.TotalProductCost), 2) AS GrossProfit,
    CASE 
        WHEN SUM(f.SalesAmount) != 0 THEN 
            ROUND((SUM(f.SalesAmount) - SUM(f.TotalProductCost)) / SUM(f.SalesAmount) * 100, 1) 
    END AS GrossMarginPct,
    CURRENT_TIMESTAMP() AS _gold_timestamp
FROM silver_fact_sales f 
JOIN silver_dim_date d ON f.OrderDateKey = d.DateKey 
LEFT JOIN silver_dim_product p ON f.ProductKey = p.ProductKey
WHERE f.IsReturn = 0 
GROUP BY d.FiscalYear, f.Channel, p.Category;

-- Log gold stage
INSERT INTO workspace.medallion_sql_pipeline.pipeline_run_log
SELECT
  DATE_FORMAT(CURRENT_TIMESTAMP(), 'yyyyMMddHHmmss') AS run_id,
  LOWER(TRIM('${mode}')) AS run_mode,
  'gold' AS stage,
  0 AS rows_in,
  COUNT(*) AS rows_out,
  'SUCCESS' AS status,
  '' AS error_msg,
  CURRENT_TIMESTAMP() AS started_at,
  CURRENT_TIMESTAMP() AS finished_at
FROM gold_channel_compare;

SELECT 'Channel compare created' AS status, COUNT(*) AS rows FROM gold_channel_compare;

In [0]:
%sql
-- Display pipeline summary
SELECT '════════════════════════════════════════' AS summary;
SELECT 'PIPELINE COMPLETED' AS status;
SELECT '════════════════════════════════════════' AS summary;

SELECT 'Bronze' AS layer, 'bronze_customer' AS table_name, COUNT(*) AS row_count FROM workspace.medallion_sql_pipeline.bronze_customer
UNION ALL
SELECT 'Bronze', 'bronze_product', COUNT(*) FROM workspace.medallion_sql_pipeline.bronze_product
UNION ALL
SELECT 'Bronze', 'bronze_reseller', COUNT(*) FROM workspace.medallion_sql_pipeline.bronze_reseller
UNION ALL
SELECT 'Bronze', 'bronze_sales_territory', COUNT(*) FROM workspace.medallion_sql_pipeline.bronze_sales_territory
UNION ALL
SELECT 'Bronze', 'bronze_date', COUNT(*) FROM workspace.medallion_sql_pipeline.bronze_date
UNION ALL
SELECT 'Bronze', 'bronze_sales_order', COUNT(*) FROM workspace.medallion_sql_pipeline.bronze_sales_order
UNION ALL
SELECT 'Bronze', 'bronze_sales', COUNT(*) FROM workspace.medallion_sql_pipeline.bronze_sales
UNION ALL
SELECT 'Silver', 'silver_dim_customer', COUNT(*) FROM workspace.medallion_sql_pipeline.silver_dim_customer
UNION ALL
SELECT 'Silver', 'silver_dim_product', COUNT(*) FROM workspace.medallion_sql_pipeline.silver_dim_product
UNION ALL
SELECT 'Silver', 'silver_dim_reseller', COUNT(*) FROM workspace.medallion_sql_pipeline.silver_dim_reseller
UNION ALL
SELECT 'Silver', 'silver_dim_territory', COUNT(*) FROM workspace.medallion_sql_pipeline.silver_dim_territory
UNION ALL
SELECT 'Silver', 'silver_dim_date', COUNT(*) FROM workspace.medallion_sql_pipeline.silver_dim_date
UNION ALL
SELECT 'Silver', 'silver_dim_salesorder', COUNT(*) FROM workspace.medallion_sql_pipeline.silver_dim_salesorder
UNION ALL
SELECT 'Silver', 'silver_fact_sales', COUNT(*) FROM workspace.medallion_sql_pipeline.silver_fact_sales
UNION ALL
SELECT 'Gold', 'gold_sales_by_month', COUNT(*) FROM workspace.medallion_sql_pipeline.gold_sales_by_month
UNION ALL
SELECT 'Gold', 'gold_product_ranking', COUNT(*) FROM workspace.medallion_sql_pipeline.gold_product_ranking
UNION ALL
SELECT 'Gold', 'gold_customer_summary', COUNT(*) FROM workspace.medallion_sql_pipeline.gold_customer_summary
UNION ALL
SELECT 'Gold', 'gold_channel_compare', COUNT(*) FROM workspace.medallion_sql_pipeline.gold_channel_compare
ORDER BY layer, table_name;